# Notebook 05: Explainability (XAI)
## SHAP Values → Wavelength-to-Biology Mapping
**Novelty:** First paper to explain *why* UV-Vis detects contamination at specific wavelengths

In [ ]:
import numpy as np, pandas as pd, struct
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import shap
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
import warnings; warnings.filterwarnings('ignore')

BASE = Path('/run/media/sham/AI_/ai-stack/projects/biopharma-contamination-detection')
OUT = BASE / 'data' / 'processed'
FIG = BASE / 'figures'; FIG.mkdir(exist_ok=True)
df = pd.read_parquet(OUT / 'real_dataset.parquet')

In [ ]:
def unpack(row):    n = row['n_wl']; data = struct.unpack(f'{n*2}d', row['spectrum_bytes'])    return np.array(data[::2]), np.array(data[1::2])target_wl = np.arange(230, 610, 1)def interp(row): return np.interp(target_wl, *unpack(row))X = np.vstack(df.apply(interp, axis=1).values)y = df['label'].valuesscaler = RobustScaler().fit(X[y==0])X_s = scaler.transform(X)print(f'Feature matrix: {X_s.shape} (wavelengths as features)')

## Train Random Forest for SHAP analysis

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_s, y)
print(f'RF accuracy: {rf.score(X_s, y):.4f}')

## SHAP Analysis

In [ ]:
# Use a subset for SHAP (computationally expensive)
X_sample = X_s[:200]
y_sample = y[:200]
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_sample)
print(f'SHAP values shape: {shap_values.shape}')

## SHAP Summary Plot

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
shap.summary_plot(shap_values[:, :, 1] if shap_values.ndim == 3 else shap_values[:, :, -1],
    X_sample, feature_names=[f'{wl}nm' for wl in target_wl],
    show=False, plot_size=None, max_display=20)
plt.tight_layout()
fig.savefig(FIG / 'shap_summary.png', dpi=300, bbox_inches='tight')
print('Saved: shap_summary.png')
plt.close()

## Wavelength Importance Map

In [ ]:
shap_1d = np.abs(shap_values[:, :, 1] if shap_values.ndim == 3 else shap_values[:, :, -1]).mean(axis=0)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(target_wl, shap_1d, linewidth=1.5, color='#E91E63')
ax.fill_between(target_wl, shap_1d, alpha=0.3, color='#E91E63')
ax.axvline(260, color='red', alpha=0.4, linestyle='--', label='DNA (260nm)')
ax.axvline(280, color='blue', alpha=0.4, linestyle='--', label='Protein (280nm)')
ax.axvline(430, color='orange', alpha=0.4, linestyle='--', label='pH indicator (430nm)')
ax.axvline(600, color='green', alpha=0.4, linestyle='--', label='Biomass OD600')
ax.set_xlabel('Wavelength (nm)', fontsize=12)
ax.set_ylabel('Mean |SHAP Value| (Feature Importance)', fontsize=12)
ax.set_title('Wavelength Importance: Which Spectral Regions Drive Detection?', fontsize=14)
ax.legend(fontsize=10)
plt.tight_layout()
fig.savefig(FIG / 'wavelength_importance.png', dpi=300)
print('Saved: wavelength_importance.png')
plt.close()

top5 = np.argsort(shap_1d)[-5:][::-1]
print(f'\nTop 5 most important wavelengths:')
for idx in top5:
    print(f'  {target_wl[idx]}nm: SHAP={shap_1d[idx]:.6f}')

## Biological Interpretation

In [ ]:
print('\nBIOLOGICAL INTERPRETATION')
print('='*50)
bio_map = {'DNA/RNA': 260, 'Protein (aromatic AA)': 280, 'Phenol red (pH)': 430, 'Biomass/turbidity': 600}
for bio, wl in bio_map.items():
    idx = np.argmin(np.abs(target_wl - wl))
    print(f'  {wl}nm → {bio}: SHAP={shap_1d[idx]:.6f}')
print('\n✅ Explainability analysis complete!')